# Extracting source code with TorchLens

TorchLens captures the call stack and the actual line of source code for every
operation in a forward pass. This notebook shows how to enable source-code
capture and access the resulting metadata.


In [ ]:
import sys

sys.path.insert(0, ".")

import torch
import torchlens as tl
from _demo_models import TinyMLP

## 1. Trace with `save_code_context=True`

The source-code metadata is opt-in (it costs a small `inspect`-based stack walk
per op at capture time). Pass `save_code_context=True` to `tl.trace()`:


In [ ]:
torch.manual_seed(0)
model = TinyMLP()
x = torch.rand(2, 8)
trace = tl.trace(model, x, save_code_context=True)

print(f"Captured {trace.num_ops} ops across {len(trace.layer_labels)} layers.")

## 2. Inspect a single op's source location

Each op carries a `code_context` field — a list of `FuncCallLocation` frames
representing the call stack that led to that op. The first frame is typically
the user's own model code; later frames are torch internals. (Input/output
boundary ops have an empty `code_context`.)


In [ ]:
op = trace["linear_1_1"]
frame = op.code_context[0]

print(f"Function:    {frame.code_qualname}")
print(f"File:        {frame.file}")
print(f"Line:        {frame.line_number}")
print(f"Source line: {frame.call_line.strip()!r}")

## 3. Map every op back to its originating source line

A common use case is to walk the trace and print where each op was created.
We skip the input/output boundary ops which carry no source context.


In [ ]:
for op_label in trace.op_labels:
    op = trace[op_label]
    if not op.code_context:
        continue  # boundary ops (input/output) have no source context
    frame = op.code_context[0]
    location = f"{frame.file.split('/')[-1]}:{frame.line_number}"
    print(f"  {op_label:<20s}  ({location})  {frame.call_line.strip()}")

## 4. Inspect the full call stack of a single op

For deeper debugging, the full stack is available. The first frame is the
user code; intermediate frames typically include torch's nn module machinery;
the deepest frame is the eager-mode dispatch site.


In [ ]:
op = trace["linear_1_1"]
print(f"Stack for {op.layer_label} ({op.func_name}):\n")
for index, frame in enumerate(op.code_context):
    qualname = frame.code_qualname or frame.func_name
    print(f"  [{index}] {qualname}")
    print(f"      {frame.file}:{frame.line_number}")
    if frame.call_line.strip():
        print(f"      | {frame.call_line.strip()}")

## 5. Build a model-source-to-op index

Group ops by the source line that produced them — useful for "which line of
my model's `forward()` produced which tensors?" debugging.


In [ ]:
from collections import defaultdict

ops_by_source = defaultdict(list)
for op_label in trace.op_labels:
    op = trace[op_label]
    if not op.code_context:
        continue
    user_frame = op.code_context[0]  # first frame = user code
    key = (user_frame.file, user_frame.line_number, user_frame.call_line.strip())
    ops_by_source[key].append(op_label)

for (file, line, src), op_labels in ops_by_source.items():
    print(f"{file.split('/')[-1]}:{line}  -->  {', '.join(op_labels)}")
    print(f"  | {src}\n")

---

That's it — `save_code_context=True` plus walking `op.code_context` covers the
full source-attribution story. The same `FuncCallLocation` records survive
portable save/load, so a saved Trace can be reopened later and still report
where each op originated.
